In [8]:
import pandas as pd
import numpy as np
import csv
import json
import glob
import os

In [9]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)

In [ ]:
class Preprocessor:
    def extract_gpu(subject: str):
        """Возвращает 0 — встроенная, 1 — дискретная"""
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject
    
        # Встроенная
        if re.search(r'mac\s*book', s, re.IGNORECASE):
            return 0
        if re.search(r'Intel\s+Iris\s+Xe', s, re.IGNORECASE):
            return 0
        if re.search(r'Intel\s+UHD\s+Graphics', s, re.IGNORECASE):
            return 0
    
        # Дискретная
        if re.search(r'(?:NVIDIA|GeForce|RTX|GTX|MX|Quadro|Arc|Radeon)', s, re.IGNORECASE):
            return 1

        return float('nan')

    def extract_gpu_model(subject: str):
    
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject    
        if pd.isna(subject):
            return float('nan')
    
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject
    
        def find_mem(text: str):
            """
            Берёт VRAM сразу после модели.
            Если следом идёт ЕЩЁ одна память (16ГБ 1000ГБ) — это RAM+SSD, пропускаем.
            Если число > 64 — это точно не VRAM.
            """
            m = re.search(r'(\d+)\s*(?:ГБ|GB)', text, re.IGNORECASE)
            if not m:
                return float('nan')
            if int(m.group(1)) > 64:
                return float('nan')
            tail = text[m.end():m.end() + 25]
            if re.search(r'\d+\s*(?:ГБ|GB|ТБ|TB)', tail, re.IGNORECASE):
                return float('nan')
            return m.group(0)
    
        def fmt(base: str, mem: str):
            return f"{base} {mem}" if mem else base
    
        SEP = r'[\s\-]*'   # separator: space, - or nothing
    
        if re.search(r'mac\s*book', s, re.IGNORECASE):
            return 0
        # Intel 
        m = re.search(r'Intel\s+Iris\s+Xe\s+Graphics\s+(G\d+)', s, re.IGNORECASE)
        if m:
            return f"Intel Iris XE Graphics {m.group(1).upper()}"
        if re.search(r'Intel\s+Iris\s+Xe', s, re.IGNORECASE):
            return 'Intel Iris XE Graphics'
        if re.search(r'Intel\s+UHD\s+Graphics', s, re.IGNORECASE):
            return 'Intel UHD Graphics'
        m = re.search(r'(?:Intel\s+)?Arc\s+(A\d{3}M?)', s, re.IGNORECASE)
        if m:
            return fmt(f"Intel Arc {m.group(1).upper()}", find_mem(s[m.end():]))
    
        # NVIDIA Quadro 
        m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}RTX{SEP}(\d{{3,5}})\b', s, re.IGNORECASE)
        if m:
            return fmt(f"NVIDIA Quadro RTX {m.group(1)}", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}T{SEP}(\d{{3,4}})\s*(Max-Q)?', s, re.IGNORECASE)
        if m:
            maxq = " Max-Q" if m.group(2) else ""
            return fmt(f"NVIDIA Quadro T{m.group(1)}{maxq}", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}P{SEP}(\d{{3,4}})\b', s, re.IGNORECASE)
        if m:
            return fmt(f"NVIDIA Quadro P{m.group(1)}", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}M{SEP}(\d{{3,4}})\b', s, re.IGNORECASE)
        if m:
            return fmt(f"NVIDIA Quadro M{m.group(1)}", find_mem(s[m.end():]))
    
        # NVIDIA GeForce RTX / GTX / MX 
        m = re.search(
            rf'(?:NVIDIA\s+)?(?:GeForce\s+)?RTX{SEP}(\d{{4}}){SEP}(Ti\b)?\s*(Max-Q)?',
            s, re.IGNORECASE
        )
        if m:
            ti   = " Ti"    if m.group(2) else ""
            maxq = " Max-Q" if m.group(3) else ""
            return fmt(f"NVIDIA GeForce RTX {m.group(1)}{ti}{maxq}", find_mem(s[m.end():]))
    
        m = re.search(
            rf'(?:NVIDIA\s+)?(?:GeForce\s+)?GTX{SEP}(\d{{4}}){SEP}(Ti\b)?\s*(Max-Q)?',
            s, re.IGNORECASE
        )
        if m:
            ti   = " Ti"    if m.group(2) else ""
            maxq = " Max-Q" if m.group(3) else ""
            return fmt(f"NVIDIA GeForce GTX {m.group(1)}{ti}{maxq}", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:NVIDIA\s+)?(?:GeForce\s+)?MX{SEP}(\d{{3,4}})', s, re.IGNORECASE)
        if m:
            return fmt(f"NVIDIA GeForce MX{m.group(1)}", find_mem(s[m.end():]))
    
        # AMD Radeon
        m = re.search(r'(?:AMD\s+)?Radeon\s+RX\s+Vega\s+M\s+GL', s, re.IGNORECASE)
        if m:
            return fmt("AMD Radeon RX Vega M GL", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:AMD\s+)?Radeon\s+Pro{SEP}(\d{{3,4}}[A-Z]*)', s, re.IGNORECASE)
        if m:
            return fmt(f"AMD Radeon Pro {m.group(1)}", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:AMD\s+)?(?:Radeon\s+)?RX{SEP}(\d{{3,4}}M?)', s, re.IGNORECASE)
        if m:
            return fmt(f"AMD Radeon RX {m.group(1).upper()}", find_mem(s[m.end():]))
    
        m = re.search(rf'(?:AMD\s+)?Radeon{SEP}(\d{{3,4}}[A-Z]*)', s, re.IGNORECASE)
        if m:
            return fmt(f"AMD Radeon {m.group(1).upper()}", find_mem(s[m.end():]))

    def extract_ram(subject):
    """Извлекает объём RAM в формате 'X ГБ'"""
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    # 1. Явный маркер RAM/ОЗУ
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s*(?:RAM|ОЗУ)', s, re.IGNORECASE)
    if m:
        return f"{m.group(1)} ГБ"
    m = re.search(r'(?:RAM|ОЗУ)[\s:\-]*(\d+)\s*(?:ГБ|GB)?', s, re.IGNORECASE)
    if m:
        return f"{m.group(1)} ГБ"

    # 2. Apple-style "16/256" — первое число RAM, второе ROM
    m = re.search(r'\b(\d{1,3})\s*/\s*(\d{2,4})\b', s)
    if m:
        ram, rom = int(m.group(1)), int(m.group(2))
        if ram in {2,3,4,6,8,12,16,24,32,48,64,128} and rom >= 64:
            return f"{ram} ГБ"

    # 3. "16GB/512GB" — первое RAM
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s*/\s*\d+\s*(?:ГБ|GB|ТБ|TB)', s, re.IGNORECASE)
    if m and int(m.group(1)) <= 128:
        return f"{m.group(1)} ГБ"

    # 4. "16ГБ 1000ГБ" — два числа подряд, первое RAM
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s+(\d+)\s*(?:ГБ|GB|ТБ|TB)', s, re.IGNORECASE)
    if m and int(m.group(1)) <= 128:
        return f"{m.group(1)} ГБ"

    return float('nan')


    def extract_rom(subject):
        """Извлекает объём ROM в формате 'X ГБ'"""
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject
    
        # 1. ТБ → переводим в ГБ
        m = re.search(r'(\d+)\s*(?:ТБ|TB)', s, re.IGNORECASE)
        if m:
            return f"{int(m.group(1)) * 1000} ГБ"
    
        # 2. SSD/HDD маркер
        m = re.search(r'(?:SSD|HDD)[\s\-]*(\d+)\s*(?:ГБ|GB)', s, re.IGNORECASE)
        if m:
            return f"{m.group(1)} ГБ"
        m = re.search(r'(\d+)\s*(?:ГБ|GB)\s*(?:SSD|HDD)', s, re.IGNORECASE)
        if m:
            return f"{m.group(1)} ГБ"
    
        # 3. Apple-style "16/256" — второе число ROM
        m = re.search(r'\b(\d{1,3})\s*/\s*(\d{2,4})\b', s)
        if m:
            ram, rom = int(m.group(1)), int(m.group(2))
            if ram in {2,3,4,6,8,12,16,24,32,48,64,128} and rom >= 64:
                return f"{rom} ГБ"
    
        # 4. "16GB/512GB" — второе число ROM
        m = re.search(r'\d+\s*(?:ГБ|GB)\s*/\s*(\d+)\s*(?:ГБ|GB)', s, re.IGNORECASE)
        if m and int(m.group(1)) >= 64:
            return f"{m.group(1)} ГБ"
    
        # 5. "16ГБ 1000ГБ" — два числа подряд, второе ROM
        m = re.search(r'(\d+)\s*(?:ГБ|GB)\s+(\d+)\s*(?:ГБ|GB)', s, re.IGNORECASE)
        if m and int(m.group(1)) <= 128 and int(m.group(2)) >= 64:
            return f"{m.group(2)} ГБ"
    
        return float('nan')

    def extract_diagonal(subject):
    """Извлекает диагональ в дюймах (float)"""
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject
    
        # 1. С маркером дюймов: 15.6", 17,3'', 13.3 дюйм
        m = re.search(r'(\d{2}[,.]?\d?)\s*(?:[\'"]{1,2}|дюйм|inch)', s, re.IGNORECASE)
        if m:
            val = float(m.group(1).replace(',', '.'))
            if 10 <= val <= 18:
                return val
    
        # 2. Десятичное число типа 15.6, 17.3, 13.3 — редко false positive
        m = re.search(r'\b(1[0-7][,.]\d)\b', s)
        if m:
            val = float(m.group(1).replace(',', '.'))
            if 10 <= val <= 18:
                return val
    
        # 3. После MacBook/Pro/Air — целое число 13-17
        m = re.search(r'(?:MacBook|Air|Pro)\s+(\d{2})\b', s, re.IGNORECASE)
        if m:
            val = float(m.group(1))
            if 10 <= val <= 18:
                return val
    
        return float('nan')
    def extract_processor(subject):
    """Извлекает категорию процессора"""
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject
    
        # Apple M1/M2/M3/M4/M5 (Pro/Max/Ultra)
        m = re.search(r'\bM([1-5])\s*(Pro|Max|Ultra)?\b', s)
        # Только если контекст Apple/MacBook
        if m and re.search(r'mac\s*book|apple', s, re.IGNORECASE):
            suffix = f" {m.group(2)}" if m.group(2) else ""
            return f"Apple M{m.group(1)}{suffix}"
    
        # Intel Core iN
        m = re.search(r'(?:Core\s+)?i([3579])[\s\-]?\d{3,5}[A-Z]{0,2}', s, re.IGNORECASE)
        if m:
            return f"Intel Core i{m.group(1)}"
        m = re.search(r'Core\s+i([3579])', s, re.IGNORECASE)
        if m:
            return f"Intel Core i{m.group(1)}"
    
        # AMD Ryzen — "Ryzen 5", "R5-7520U"
        m = re.search(r'Ryzen\s+([3579])', s, re.IGNORECASE)
        if m:
            return f"AMD Ryzen {m.group(1)}"
        m = re.search(r'\bR([3579])\s*[\-–]\s*\d{3,5}[A-Z]{0,2}', s, re.IGNORECASE)
        if m:
            return f"AMD Ryzen {m.group(1)}"
    
        # Intel Pentium / Celeron
        if re.search(r'Pentium', s, re.IGNORECASE):
            return 'Intel Pentium'
        if re.search(r'Celeron', s, re.IGNORECASE):
            return 'Intel Celeron'
    
        return float('nan')

    def extract_brand(subject):
        if pd.isna(subject) or not isinstance(subject, str):
            return float('nan')
        s = subject
    
        brand_patterns = [
            (r'\bMacBook|Apple\b',                  'Apple'),
            (r'\bASUS|Aorus\b',                     'Asus'), 
            (r'\bLenovo|ThinkPad|IdeaPad|Legion\b', 'Lenovo'),
            (r'\bHP|HewlettPackard|Pavilion|EliteBook|ProBook|Omen|Victus\b', 'HP'),
            (r'\bDell|Inspiron|Latitude|XPS|Alienware|Vostro\b', 'Dell'),
            (r'\bAcer|Aspire|Predator|Nitro|Swift|TravelMate\b', 'Acer'),
            (r'\bMSI\b',                            'MSI'),
            (r'\bHuawei|MateBook\b',                'Huawei'),
            (r'\bHonor\s*MagicBook|Honor\b',        'Honor'),
            (r'\bSamsung|Galaxy\s*Book\b',          'Samsung'),
            (r'\bXiaomi|RedmiBook|Redmi\b',         'Xiaomi'),
            (r'\bGigabyte\b',                       'Gigabyte'),
            (r'\bRazer\b',                          'Razer'),
            (r'\bMicrosoft|Surface\b',              'Microsoft'),
            (r'\bLG\s*Gram|LG\b',                   'LG'),
            (r'\bToshiba|Dynabook\b',               'Toshiba'),
            (r'\bSony|VAIO\b',                      'Sony'),
            (r'\bChuwi\b',                          'Chuwi'),
            (r'\bThunderobot\b',                    'Thunderobot'),
            (r'\bMechRevo\b',                       'MechRevo'),
            (r'\bMaibenben\b',                      'Maibenben'),
            (r'\bDigma\b',                          'Digma'),
            (r'\bIRBIS\b',                          'IRBIS'),
            (r'\bHaier\b',                          'Haier'),
        ]
    
        for pattern, brand in brand_patterns:
            if re.search(pattern, s, re.IGNORECASE):
                return brand
    
        return float('nan')
        

In [ ]:
    def parse_number(val):
        if pd.isna(val):
            return np.nan
        match = re.search(r'\d+\.?\d*', str(val))
        return float(match.group()) if match else np.nan
    
    def extract_first_number(series: pd.Series, transform=None) -> pd.Series:
        result = series.apply(parse_number)
        return result
    
    df["rom_volume"] = extract_first_number(df["rom_volume"])
    df["ram_volume"] = extract_first_number(df["ram_volume"])
    df["diagonal"] = extract_first_number(df["diagonal"])